---
### From Observation to Exploration

In Session 2, you watched a 3-tool agent answer a well-defined clinical question. You traced its reasoning and compared it to your manual steps from Session 1.

Now the dynamic changes:

- **More tools** — 6 instead of 3, covering medications, encounters, and full problem lists
- **Open-ended questions** — you choose what to ask, not us
- **Goal shift** — success is not just getting an answer; it is finding where the agent breaks

You are now the **quality assurance team** for a clinical AI agent. Your job is to probe its capabilities, find its limits, and document what happens when it fails. This is how real AI systems are evaluated before deployment — through structured adversarial testing by domain experts.

# INSTRUCTOR VERSION

# Session 3: Open-Ended Agent Exploration

## Your Goals Today
1. Run **at least 2** different clinical questions through the agent
2. Find **at least 1** case where the agent does something surprising or wrong
3. Document what happened and why
4. Submit your deliverable (generated automatically at the end)

The agent now has **6 tools** — 3 new ones beyond what you saw in Session 2.
It must figure out which tools to use and how. This is exploratory —
there are no "right" questions.

### 📋 Clinical Code Reference

| Code | System | Meaning | Used In | ICD-10 Reference |
|------|--------|---------|---------|------------------|
| 44054006 | SNOMED CT | Type 2 Diabetes Mellitus | Condition search | E11 |
| 59621000 | SNOMED CT | Essential Hypertension | (Session 3) | I10 |
| 4548-4 | LOINC | Hemoglobin A1c (HbA1c) | Observation search | - |
| 85354-9 | LOINC | Blood Pressure panel | (Session 3) | - |
| 2160-0 | LOINC | Creatinine [Mass/volume] in Serum or Plasma | (Session 3) | - |

**HbA1c Interpretation:**
- < 5.7%: Normal
- 5.7% – 6.4%: Prediabetes
- ≥ 6.5%: Diabetes
- \> 7.0%: Poor glycemic control — needs intervention

**Note on threshold:** We use > 7.0% because the Synthea-generated data on this server skews toward lower HbA1c values. In clinical practice, thresholds vary by guideline (commonly 7.0%–9.0%).

In [ ]:
# Install required packages (only needed once per Colab session)
!pip install -q anthropic requests pandas

In [ ]:
# ============================================================
# SETUP — Anthropic API and FHIR Server
# ============================================================
import os, json, requests
from anthropic import Anthropic

# ---- API Key Setup ----
# Try to get API key from Colab Secrets first, then environment variable
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError("Set ANTHROPIC_API_KEY in Colab Secrets or environment")

# ---- Initialize Anthropic Client ----
client = Anthropic(api_key=api_key)
MODEL = "claude-sonnet-4-20250514"

# ---- FHIR Server ----
FHIR_BASE = "https://launch.smarthealthit.org/v/r4/fhir"

print(f"✅ LLM: Anthropic Claude ({MODEL})")
print(f"✅ FHIR server: {FHIR_BASE}")

In [ ]:

# ============================================================
# FHIR TOOL FUNCTIONS
# ============================================================
# These are the same queries you ran manually in Session 1,
# now packaged as reusable functions.

def search_conditions(code: str, max_results: int = 20) -> dict:
    """Search for Condition resources by diagnosis code (SNOMED CT or ICD-10)."""
    resp = requests.get(f"{FHIR_BASE}/Condition",
        params={"code": code, "_count": max_results, "_format": "json"},
        timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        results.append({
            "condition_id": r.get("id", ""),
            "patient_reference": r.get("subject", {}).get("reference", ""),
            "code": coding.get("code", ""),
            "code_display": coding.get("display", ""),
            "onset": r.get("onsetDateTime", "unknown")
        })
    return {"total": bundle.get("total", len(results)), "results": results}


def get_patient(patient_id: str) -> dict:
    """Retrieve a single Patient resource by FHIR ID. Returns demographics."""
    resp = requests.get(f"{FHIR_BASE}/Patient/{patient_id}",
        params={"_format": "json"}, timeout=15)
    resp.raise_for_status()
    p = resp.json()
    name = p.get("name", [{}])[0]
    return {
        "id": p.get("id", patient_id),
        "name": f"{' '.join(name.get('given', []))} {name.get('family', '')}".strip(),
        "birthDate": p.get("birthDate", "unknown"),
        "gender": p.get("gender", "unknown")
    }


def search_observations(patient_id: str, loinc_code: str, max_results: int = 5) -> dict:
    """Search Observations for a patient by LOINC code. Returns values sorted most recent first."""
    resp = requests.get(f"{FHIR_BASE}/Observation",
        params={
            "subject": f"Patient/{patient_id}",
            "code": loinc_code,
            "_sort": "-date",
            "_count": max_results,
            "_format": "json"
        }, timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        value_qty = r.get("valueQuantity", {})
        results.append({
            "date": r.get("effectiveDateTime", "unknown"),
            "value": value_qty.get("value", "N/A"),
            "unit": value_qty.get("unit", "")
        })
    return {"patient_id": patient_id, "loinc_code": loinc_code, "results": results}


# Quick smoke test
print("🔍 Testing FHIR tools...")
test = search_conditions("44054006", max_results=3)
print(f"   search_conditions('44054006'): {test['total']} total, {len(test['results'])} returned")
if test["results"]:
    test_pid = test["results"][0]["patient_reference"].split("/")[-1]
    test_pt = get_patient(test_pid)
    print(f"   get_patient('{test_pid}'): {test_pt['name']}")
    test_obs = search_observations(test_pid, "4548-4", max_results=1)
    print(f"   search_observations('{test_pid}', '4548-4'): {len(test_obs['results'])} results")
print("✅ All FHIR tools working")


---
### New Clinical Resources: Medications, Encounters, and Problem Lists

Session 2 used three FHIR resource types: Condition, Patient, and Observation. The cell below adds functions for three more:

- **MedicationRequest** — what drugs has the patient been prescribed? Includes medication name, code, status (active, stopped, completed), and the date it was authored.
- **Encounter** — what clinical visits has the patient had? Covers office visits, emergency department visits, and hospitalizations, with dates and status.
- **Condition (all for a patient)** — the patient's complete problem list. Unlike `search_conditions` which searches by diagnosis code across all patients, `search_all_conditions` retrieves every diagnosis for one specific patient.

Together with the original three tools, the agent can now build much richer patient profiles — but it also has more choices to make and more ways to go wrong.

In [ ]:

def search_medications(patient_id: str, max_results: int = 10) -> dict:
    """Search for MedicationRequest resources for a patient. Returns medication names and statuses."""
    resp = requests.get(f"{FHIR_BASE}/MedicationRequest",
        params={"subject": f"Patient/{patient_id}", "_count": max_results,
                "_sort": "-date", "_format": "json"}, timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        med = r.get("medicationCodeableConcept", {}).get("coding", [{}])[0]
        results.append({
            "medication": med.get("display", "unknown"),
            "code": med.get("code", ""),
            "status": r.get("status", "unknown"),
            "authored_on": r.get("authoredOn", "unknown")
        })
    return {"patient_id": patient_id, "results": results}


def search_encounters(patient_id: str, max_results: int = 10) -> dict:
    """Search for Encounter resources (visits, hospitalizations) for a patient."""
    resp = requests.get(f"{FHIR_BASE}/Encounter",
        params={"subject": f"Patient/{patient_id}", "_count": max_results,
                "_sort": "-date", "_format": "json"}, timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        enc_type = r.get("type", [{}])[0].get("coding", [{}])[0]
        period = r.get("period", {})
        results.append({
            "encounter_type": enc_type.get("display", "unknown"),
            "class": r.get("class", {}).get("code", "unknown"),
            "status": r.get("status", "unknown"),
            "period_start": period.get("start", "unknown"),
            "period_end": period.get("end", "unknown")
        })
    return {"patient_id": patient_id, "results": results}


def search_all_conditions(patient_id: str, max_results: int = 20) -> dict:
    """Search for ALL Condition resources for a patient (full problem list)."""
    resp = requests.get(f"{FHIR_BASE}/Condition",
        params={"subject": f"Patient/{patient_id}", "_count": max_results,
                "_format": "json"}, timeout=15)
    resp.raise_for_status()
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        clinical_status = r.get("clinicalStatus", {}).get("coding", [{}])[0]
        results.append({
            "condition": coding.get("display", "unknown"),
            "code": coding.get("code", ""),
            "system": coding.get("system", ""),
            "onset": r.get("onsetDateTime", "unknown"),
            "clinical_status": clinical_status.get("code", "unknown")
        })
    return {"patient_id": patient_id, "results": results}


In [ ]:
# Quick test of the new tools
print("🔍 Testing new tools...")
# Use a patient ID from the condition search above
if test_pid:
    meds = search_medications(test_pid, max_results=3)
    print(f"   search_medications: {len(meds['results'])} results")
    encs = search_encounters(test_pid, max_results=3)
    print(f"   search_encounters: {len(encs['results'])} results")
    conds = search_all_conditions(test_pid, max_results=5)
    print(f"   search_all_conditions: {len(conds['results'])} results")
    print("✅ All 6 tools working")
else:
    print("⚠️  No test patient available — check search_conditions above")

---
### What 6 Tools Can Do That 3 Could Not

With 3 tools the agent could answer: "Find patients with diagnosis X and get their lab value Y." That is a narrow but well-defined pipeline.

With 6 tools, the space of answerable questions expands significantly:

- What medications are diabetic patients taking?
- How often do hypertensive patients visit the ER?
- What is the full problem list for patients with high HbA1c?
- Are patients with both diabetes and hypertension on appropriate medications?

But more capability brings more complexity. The agent must now decide **which** of 6 tools to use, not just **when** to use one of 3. Watch for:

- **Suboptimal tool choice** — using `search_conditions` (by code, all patients) when `search_all_conditions` (all conditions, one patient) was more appropriate, or vice versa
- **Missed tools** — not using a relevant tool at all
- **Unnecessary calls** — fetching data that does not help answer the question

In [ ]:
# ============================================================
# TOOL SCHEMAS — Define available tools for the agent
# ============================================================
# Tool schemas tell Claude what functions are available and how to use them.
# Claude uses Anthropic's native tool format.

tools = [
    {
        "name": "search_conditions",
        "description": "Search for patient Condition resources on the FHIR server by diagnosis code (SNOMED CT or ICD-10). Returns a list of conditions with patient references, codes, and onset dates. Use this to find patients with a specific diagnosis.",
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "Diagnosis code to search for. Examples: '44054006' for Type 2 diabetes (SNOMED CT), '59621000' for hypertension (SNOMED CT). Also accepts ICD-10 codes like 'E11' or 'I10'."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of condition entries to return. Default 20.",
                    "default": 20
                }
            },
            "required": ["code"]
        }
    },
    {
        "name": "get_patient",
        "description": "Retrieve a single Patient resource by their FHIR patient ID. Returns demographics including full name, birth date, and gender. Use this after getting a patient reference from another resource.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient resource ID (the part after 'Patient/' in a reference). Example: 'abc123'"
                }
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "search_observations",
        "description": "Search for Observation resources (lab results, vital signs) for a specific patient by LOINC code. Returns values sorted by date with most recent first. Common LOINC codes: '4548-4' for HbA1c, '85354-9' for blood pressure, '2160-0' for creatinine.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient ID to search observations for"
                },
                "loinc_code": {
                    "type": "string",
                    "description": "LOINC code for the observation type. Examples: '4548-4' (HbA1c), '85354-9' (blood pressure), '2160-0' (creatinine)"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return. Default 5.",
                    "default": 5
                }
            },
            "required": ["patient_id", "loinc_code"]
        }
    }
]

available_functions = {
    "search_conditions": search_conditions,
    "get_patient": get_patient,
    "search_observations": search_observations,
}

print("✅ Tool schemas defined (3 tools)")
print()
for t in tools:
    params = ", ".join(t["input_schema"].get("required", []))
    print(f"   • {t['name']}({params})")
    print(f"     {t['description'][:80]}...")
    print()

In [ ]:
# ============================================================
# ADD SESSION 3 TOOLS — Three additional tools
# ============================================================

# Initialize base tools (self-contained if TOOL SCHEMAS cell was skipped)
tools = [
    {
        "name": "search_conditions",
        "description": "Search for patient Condition resources on the FHIR server by diagnosis code (SNOMED CT or ICD-10). Returns a list of conditions with patient references, codes, and onset dates. Use this to find patients with a specific diagnosis.",
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "Diagnosis code to search for. Examples: '44054006' for Type 2 diabetes (SNOMED CT), '59621000' for hypertension (SNOMED CT). Also accepts ICD-10 codes like 'E11' or 'I10'."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of condition entries to return. Default 20.",
                    "default": 20
                }
            },
            "required": ["code"]
        }
    },
    {
        "name": "get_patient",
        "description": "Retrieve a single Patient resource by their FHIR patient ID. Returns demographics including full name, birth date, and gender. Use this after getting a patient reference from another resource.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient resource ID (the part after 'Patient/' in a reference). Example: 'abc123'"
                }
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "search_observations",
        "description": "Search for Observation resources (lab results, vital signs) for a specific patient by LOINC code. Returns values sorted by date with most recent first. Common LOINC codes: '4548-4' for HbA1c, '85354-9' for blood pressure, '2160-0' for creatinine.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient ID to search observations for"
                },
                "loinc_code": {
                    "type": "string",
                    "description": "LOINC code for the observation type. Examples: '4548-4' (HbA1c), '85354-9' (blood pressure), '2160-0' (creatinine)"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return. Default 5.",
                    "default": 5
                }
            },
            "required": ["patient_id", "loinc_code"]
        }
    }
]

available_functions = {
    "search_conditions": search_conditions,
    "get_patient": get_patient,
    "search_observations": search_observations,
}

# Add the three new tools for Session 3
tools.extend([
    {
        "name": "search_medications",
        "description": "Search for MedicationRequest resources for a patient. Returns medication names, codes, statuses, and dates. Use this to find what medications a patient has been prescribed.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient ID"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results. Default 10.",
                    "default": 10
                }
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "search_encounters",
        "description": "Search for Encounter resources (office visits, emergency visits, hospitalizations) for a patient. Returns encounter types, dates, and statuses. Use this to understand a patient's visit history.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient ID"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results. Default 10.",
                    "default": 10
                }
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "search_all_conditions",
        "description": "Search for ALL Condition resources for a specific patient (their complete problem list). Returns condition names, codes, onset dates, and clinical status. Use this to get a patient's full medical history of diagnoses.",
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The FHIR Patient ID"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results. Default 20.",
                    "default": 20
                }
            },
            "required": ["patient_id"]
        }
    }
])

# Add the new functions to available_functions
available_functions.update({
    "search_medications": search_medications,
    "search_encounters": search_encounters,
    "search_all_conditions": search_all_conditions,
})

print("✅ Tool schemas updated (6 tools total)")
print()
for t in tools:
    params = ", ".join(t["input_schema"].get("required", []))
    print(f"   • {t['name']}({params})")

---
### Same System Prompt, More Tools

The system prompt below is identical to Session 2 — the same behavioral instructions, strategy template, and constraints. The only change is that the agent now sees 6 tool schemas instead of 3.

This is an intentional design choice. By keeping the system prompt constant, you can isolate the effect of adding tools. Any differences in behavior come from the expanded tool set, not from new instructions.

Watch for whether the system prompt's five-step strategy still works with 6 tools. The original steps (identify condition, extract references, get demographics, look up observations, synthesize) assumed the 3-tool setup. With medications and encounters now available, the agent may need to deviate from that template — and the system prompt does not explicitly guide it on when to use the new tools.

This is a common real-world challenge: system prompts that were adequate for a small tool set may become insufficient as capabilities grow.

In [ ]:
# ============================================================
# SYSTEM PROMPT — Tells the agent HOW to think
# ============================================================

SYSTEM_PROMPT = """You are a clinical data assistant with access to a FHIR server
containing synthetic patient data.

When asked a clinical question, use the available tools to query the FHIR server
and build up the data needed to answer. Think step by step:

1. First, identify what diagnosis or condition is relevant and search for it
2. Extract patient references from the conditions found
3. Retrieve patient demographics (name, birthdate, gender) for each patient
4. Look up relevant observations (lab values, vitals) for each patient
5. Synthesize a clear, accurate summary based ONLY on the data you retrieved

Rules:
- NEVER invent or assume data that was not returned by a tool call
- If a query returns no results, state that explicitly
- Always identify patients by name when demographics are available
- When comparing values to clinical thresholds, show the actual values
- Be concise but thorough — include all relevant findings"""

print("📝 System prompt defined")
print()
print("--- SYSTEM PROMPT ---")
print(SYSTEM_PROMPT)


---
### Inside the Agent Loop: How `run_agent()` Works

Session 2 introduced the agent loop at a high level: send a question, get back a tool call or a final answer, repeat. Now that you have watched the loop run and analyzed its trace, here is what the code below actually does at each phase. This is the Reason-Act-Observe cycle at the implementation level.

**Initialization.** The function starts by building a `messages` list with one entry: the user's clinical question. This list is the agent's entire memory — like a running medical chart that accumulates notes with every interaction. Nothing persists between calls except what is in this list.

**The decision point.** On each iteration, `client.messages.create(model=..., system=system_prompt, tools=tools, messages=messages)` sends the full conversation history plus all tool schemas to Claude. Claude reads everything and returns a response whose `stop_reason` is either `"tool_use"` (it wants to call a function) or `"end_turn"` (it is done reasoning and has a text answer).

**Tool execution.** When `stop_reason` is `"tool_use"`, the code extracts the function name and arguments from the response, looks up the matching Python function in `available_functions[fn_name]`, and calls it: `available_functions[fn_name](**fn_args)`. The LLM never touches the FHIR server directly — your local code does, and the LLM only sees the returned result.

**Feedback.** The tool's return value is appended to `messages` as a `tool_result` content block. On the next iteration, Claude sees the original question, every prior tool call and result, and decides what to do next — call another tool or produce a final answer.

**Termination.** The loop exits when Claude responds with text instead of a tool request, or when the step counter reaches `max_steps`. The `max_steps` guard prevents runaway loops — if the agent keeps calling tools without converging on an answer, the loop stops and returns whatever it has so far.

In [ ]:
# ============================================================
# AGENT LOOP — Run tool-use conversation with Claude
# ============================================================

def run_agent(question, system_prompt, tools, available_functions,
              max_steps=15, verbose=True):
    """
    Run the tool-use agent loop with Claude.

    Args:
        question: The clinical question to answer
        system_prompt: Instructions for the agent
        tools: Tool schemas in Anthropic format
        available_functions: Dict mapping function names to callables
        max_steps: Safety limit on LLM round-trips
        verbose: Print trace output

    Returns:
        (final_answer, tool_calls_log, messages)
    """
    tool_calls_log = []
    step = 0
    messages = [{"role": "user", "content": question}]

    if verbose:
        print(f"🧑‍⚕️ QUESTION: {question}\n")
        print("=" * 70)

    while step < max_steps:
        step += 1
        
        # Call Claude with tools
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        # Check if Claude wants to use tools or provide final answer
        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
        text_blocks = [b for b in response.content if b.type == "text"]

        if not tool_use_blocks:
            # No more tool calls — Claude has the final answer
            final = "\n".join(b.text for b in text_blocks)
            if verbose:
                print(f"\n{'=' * 70}")
                print(f"✅ FINAL ANSWER ({len(tool_calls_log)} tool calls):\n")
                print(final)
            return final, tool_calls_log, messages

        # Claude wants to use tools — serialize content blocks to avoid SDK issues
        assistant_content = []
        for block in response.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                assistant_content.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input
                })
        
        messages.append({"role": "assistant", "content": assistant_content})
        tool_results = []

        for block in tool_use_blocks:
            fn_name = block.name
            fn_args = block.input
            tool_calls_log.append({
                "step": step,
                "function": fn_name,
                "arguments": fn_args
            })

            if verbose:
                print(f"\n🔧 Step {step} | {fn_name}({json.dumps(fn_args)})")

            try:
                # Execute the function
                result = available_functions[fn_name](**fn_args)
                result_str = json.dumps(result, default=str)
                
                if verbose:
                    n_items = len(result.get("results", [])) if isinstance(result, dict) and "results" in result else None
                    if n_items is not None:
                        print(f"   → {n_items} items returned")
                    else:
                        preview = result_str[:100]
                        print(f"   → {preview}...")
            except Exception as e:
                result_str = json.dumps({"error": str(e)})
                if verbose:
                    print(f"   → ❌ Error: {e}")

            # Add tool result for Claude
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result_str
            })

        # Send tool results back to Claude
        messages.append({"role": "user", "content": tool_results})

    # Safety: max steps reached without final answer
    if verbose:
        print(f"\n⚠️ Reached maximum steps ({max_steps}) without final answer")
    return "Max steps reached without final answer", tool_calls_log, messages


print("✅ Agent loop ready")

---
### How to Formulate Good Clinical Questions

A good test question for the agent should be:

- **Specific enough** to evaluate whether the answer is correct — you need to know what a right answer looks like
- **Complex enough** to require multiple tool calls — single-tool questions do not test agent reasoning

A useful pattern: *"Find patients with [condition X] and get their [medications / labs / visits]."* This guarantees at least two tool types are needed.

For finding failures, try:

- **Conditions not in the code table** — ask about a disease whose SNOMED code is not listed in the tool descriptions. Will the agent guess a code, or say it does not know?
- **Vague questions** — "Which patients are the sickest?" How does the agent interpret "sickest" without a clear definition?
- **Multi-type reasoning** — questions that require combining data from conditions, labs, AND medications to answer properly

## 💡 Question Ideas

Pick one or make up your own!

**Straightforward** (agent should succeed):
- "Find patients with Type 2 diabetes and list their current medications"
- "Find patients with hypertension (SNOMED CT: 59621000) and their most recent blood pressure (LOINC 85354-9)"
- "How many encounters did diabetic patients have?"

**More complex** (agent may need creative chaining):
- "Find patients with both diabetes and hypertension — what medications are they on?"
- "Are there patients with diabetes who have had emergency encounters?"
- "Find diabetic patients with declining kidney function (creatinine LOINC 2160-0)"

**Edge cases** (agent may struggle):
- "Which patients are the sickest?" (vague)
- "Find patients at risk for diabetic complications" (requires clinical reasoning)
- "Compare treatment patterns across diabetic patients" (open-ended)

---
### Common Agent Failure Modes

As you run your questions, watch for these five patterns:

1. **Code hallucination** — the agent invents a SNOMED or LOINC code that does not exist, gets zero results, and may draw incorrect conclusions from the empty response.
2. **Incomplete search** — the agent checks only a few patients when there are many more matching the criteria, giving a partial picture.
3. **Wrong tool** — the agent uses `search_conditions` (finds patients by one diagnosis code) when `search_all_conditions` (gets all diagnoses for one patient) was needed, or vice versa.
4. **Premature answer** — the agent answers after just one or two tool calls, missing critical data that would change the conclusion.
5. **Over-fetching** — the agent makes too many calls (e.g., fetching medications, encounters, AND observations for every patient) and hits the step limit before synthesizing an answer.

Documenting these failures is the core of your deliverable. One well-analyzed failure teaches more than a dozen successful runs.

## 🔬 Question 1

In [ ]:
# ⬇️ ENTER YOUR FIRST CLINICAL QUESTION
user_question_1 = "Find patients with Type 2 diabetes and list their current medications"

assert user_question_1, "Please enter a question above"

answer_1, tool_calls_1, messages_1 = run_agent(
    question=user_question_1,
    system_prompt=SYSTEM_PROMPT,
    tools=tools,
    available_functions=available_functions
)

In [ ]:
# Trace for Question 1
tool_calls_log = tool_calls_1  # for the trace analysis code
# ============================================================
# TOOL CALL TRACE ANALYSIS
# ============================================================
print("📋 TOOL CALL SEQUENCE\n")
print(f"{'Step':<6} {'Function':<25} {'Key Arguments'}")
print("-" * 70)

for tc in tool_calls_log:
    args_summary = ", ".join(f"{k}={v}" for k, v in tc["arguments"].items())
    print(f"{tc['step']:<6} {tc['function']:<25} {args_summary}")

print(f"\n{'=' * 70}")
call_types = [tc["function"] for tc in tool_calls_log]
unique_tools = set(call_types)
print(f"Total tool calls: {len(tool_calls_log)}")
print(f"Unique tools used: {len(unique_tools)} — {', '.join(sorted(unique_tools))}")
for fn in sorted(unique_tools):
    print(f"   • {fn}: called {call_types.count(fn)} time(s)")


## 🔬 Question 2

In [ ]:
# ⬇️ ENTER YOUR SECOND CLINICAL QUESTION (try something different!)
user_question_2 = "Find patients with both diabetes and hypertension — what medications are they on?"

assert user_question_2, "Please enter a question above"

answer_2, tool_calls_2, messages_2 = run_agent(
    question=user_question_2,
    system_prompt=SYSTEM_PROMPT,
    tools=tools,
    available_functions=available_functions
)

In [ ]:
# Trace for Question 2
tool_calls_log = tool_calls_2
# ============================================================
# TOOL CALL TRACE ANALYSIS
# ============================================================
print("📋 TOOL CALL SEQUENCE\n")
print(f"{'Step':<6} {'Function':<25} {'Key Arguments'}")
print("-" * 70)

for tc in tool_calls_log:
    args_summary = ", ".join(f"{k}={v}" for k, v in tc["arguments"].items())
    print(f"{tc['step']:<6} {tc['function']:<25} {args_summary}")

print(f"\n{'=' * 70}")
call_types = [tc["function"] for tc in tool_calls_log]
unique_tools = set(call_types)
print(f"Total tool calls: {len(tool_calls_log)}")
print(f"Unique tools used: {len(unique_tools)} — {', '.join(sorted(unique_tools))}")
for fn in sorted(unique_tools):
    print(f"   • {fn}: called {call_types.count(fn)} time(s)")


## ✏️ Failure / Surprise Analysis

Describe a case where the agent did something unexpected:

**Question asked:**

"Find patients with both diabetes and hypertension — what medications are they on?"

**What you expected the agent to do:**

The agent would search for diabetes conditions (SNOMED 44054006), then search for hypertension conditions (SNOMED 59621000), intersect the patient sets, then look up medications for the overlapping patients.

**What the agent actually did:**

The agent searched for diabetes patients successfully, but the hypertension search (SNOMED 59621000) returned 0 results on this particular FHIR server. The agent reported that no hypertension patients were found rather than trying alternative codes or approaches.

**Why do you think it behaved this way?**

The FHIR test server (Synthea data) does not consistently use SNOMED CT code 59621000 for hypertension. The agent relies on the code provided in the tool description and has no fallback mechanism.

**How could you fix this?** (Better system prompt? Better tool descriptions?
Additional tools? Constraining the question?)

Better tool descriptions could list multiple code variants. A more robust system prompt could instruct the agent to try ICD-10 codes as fallback. Or add a "search by text" tool for fuzzy condition matching.

---
### Understanding Your Deliverable

The cell below collects your question runs — the questions you asked, the tool call logs, and the agent's final answers — into a JSON report with a timestamp.

To submit:

1. Run the deliverable cell below
2. Enter your student ID when prompted
3. Download the generated JSON file
4. Submit the file through the course portal

Make sure you have run at least 2 questions before generating the report. The report captures whatever runs are stored in memory, so do not restart the notebook kernel between running questions and generating the deliverable.

In [ ]:
# ============================================================
# DELIVERABLE — Generate your submission report
# ============================================================
from datetime import datetime

student_id = input("Enter your student ID: ")

report = {
    "student_id": student_id,
    "timestamp": datetime.now().isoformat(),
    "session": 3,
    "runs": []
}

# Collect Run 1
try:
    report["runs"].append({
        "question": user_question_1,
        "tool_calls": tool_calls_1,
        "num_tool_calls": len(tool_calls_1),
        "tools_used": list(set(tc["function"] for tc in tool_calls_1)),
        "final_answer": answer_1[:500] if answer_1 else ""
    })
    print(f"✅ Run 1: '{user_question_1[:60]}...'")
except NameError:
    print("⚠️  Run 1 not found")

# Collect Run 2
try:
    report["runs"].append({
        "question": user_question_2,
        "tool_calls": tool_calls_2,
        "num_tool_calls": len(tool_calls_2),
        "tools_used": list(set(tc["function"] for tc in tool_calls_2)),
        "final_answer": answer_2[:500] if answer_2 else ""
    })
    print(f"✅ Run 2: '{user_question_2[:60]}...'")
except NameError:
    print("⚠️  Run 2 not found")

filename = f"hackathon_session3_{student_id}.json"
with open(filename, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f"\n📄 Report saved: {filename}")
print(f"   Runs: {len(report['runs'])}")
print(f"\n   Download this file and submit it.")

---
### What is MCP? (Model Context Protocol)

Throughout these three sessions, we built a tool-use system with several ad-hoc choices: tool schemas in Anthropic's native format, tools running as local Python functions, and a hardcoded list of available tools.

**MCP (Model Context Protocol)** standardizes all of these:

- **Schema format** — a universal way to describe tools that works across LLM providers, not just Anthropic's format
- **Transport** — tools run as separate servers communicating via JSON-RPC over stdio or HTTP, instead of being local function calls in the same process
- **Discovery** — the LLM can ask "what tools are available?" at runtime, rather than receiving a hardcoded list

MCP turns our ad-hoc notebook approach into production architecture. With MCP, you could swap the FHIR tool server without changing any agent code, add new tools without restarting the agent, and use the same tools with different LLM providers. It is the difference between a prototype and a system designed for real-world deployment.

## ✏️ Final Reflection (Required)

1. **What surprised you most** about how the agent handled your questions?

   The agent's ability to autonomously chain tool calls is impressive, but it can only work within the tools it's given. When a SNOMED code returns no results, the agent doesn't know to try alternatives — it lacks medical coding expertise.

2. **When would you trust** an agent like this with real clinical data?
   What safeguards would you want?

   Trust requires: audit trails of every tool call, human review before clinical decisions, validated FHIR endpoints, bounded tool access (read-only), and a clinician in the loop. The agent should never be the sole decision-maker for patient care.

3. **How does tool use relate to MCP** (Model Context Protocol)?
   Based on what you've seen, what does MCP standardize that our
   notebook left ad hoc? (Think about: tool schemas, transport, discovery.)

   MCP standardizes: (a) tool schema format — our notebook used Anthropic's native format but other providers differ; (b) transport — we used direct function calls, MCP uses JSON-RPC over stdio/SSE; (c) discovery — our tools were hardcoded, MCP lets servers advertise capabilities dynamically. MCP would let us swap the FHIR tool server without changing agent code.